# VoxIntel — 03: Baseline Wav2Vec2 ASR Inference

Goal: run the **pretrained, non-fine-tuned** `facebook/wav2vec2-base-960h`
model over SLURP audio and see how it does out of the box — before any
fine-tuning. This is Experiment 2 from the project plan (Baseline ASR →
Intent pipeline): we need this baseline transcript quality (and its WER/CER)
as the reference point that fine-tuning in notebook 05 will be compared
against.

Everything here stays in the notebook on purpose — nothing gets moved to
`src/asr/` yet. We only promote code once it's stopped changing across a
few notebooks (see the project's research → refactor workflow).

We do reuse two things that already stabilized in `src/`:
- `SLURPDataset` (`src/data`) — for loading annotations
- `load_audio` (`src/utils/audio.py`) — for decoding FLAC → 16kHz mono waveform


## Cell 0 — Make sure `src` is importable

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)


Project root: c:\Users\ACER\OneDrive\Desktop\VoxIntel


## Cell 1 — Imports

In [2]:
import random

import torch
import pandas as pd
import numpy as np
import jiwer
from tqdm import tqdm
from transformers import AutoProcessor, Wav2Vec2ForCTC

from src.data import SLURPDataset
from src.utils.audio import load_audio

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

c:\Users\ACER\anaconda3\envs\torch26\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Cell 2 — Load Dataset

Using the **validation** split here rather than `train`. Reasoning: this
pretrained model isn't fine-tuned on SLURP yet, so there's no leakage risk
either way right now — but notebook 05 will fine-tune on `train`, and we
want the baseline numbers in this notebook to be directly comparable to the
fine-tuned numbers in notebook 06 on the *same* held-out audio. Establishing
that on `validation` now avoids having to redo this baseline later.

If you specifically want the `train` split instead (e.g. to sanity check
something before committing to that comparison plan), just change the
`split` argument below — nothing else in this notebook assumes one or the
other.


In [8]:
val_set = SLURPDataset(
    split="validation",
    root_dir="C:\\Users\\ACER\\OneDrive\\Desktop\\VoxIntel\\data\\raw\\slurp"
)

len(val_set)

8690

## Cell 3 — Load Wav2Vec2

In [9]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

processor = AutoProcessor.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")
model.to(DEVICE)
model.eval()

print("Model loaded:", model.config.model_type)


Using device: cuda


c:\Users\ACER\anaconda3\envs\torch26\lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ACER\.cache\huggingface\hub\models--facebook--wav2vec2-base-960h. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 212/212 [00:00<00:00, 560.29it/s, Materializing param=wav2vec2.feature_

Model loaded: wav2vec2


## Cell 4 — Transcribe One File

Manual, unrolled version first (no function yet) so every step is visible:
load audio → run through processor → forward pass → greedy-decode logits →
compare against ground truth.


In [10]:
sample = val_set[0]

waveform, sr = load_audio(sample["audio_path"], target_sr=16_000)
print(f"Loaded audio: {len(waveform)/sr:.2f}s at {sr}Hz")

inputs = processor(waveform, sampling_rate=sr, return_tensors="pt")
input_values = inputs.input_values.to(DEVICE)

with torch.no_grad():
    logits = model(input_values).logits

predicted_ids = torch.argmax(logits, dim=-1)
prediction = processor.batch_decode(predicted_ids)[0]

print("Ground Truth:", sample["transcript"])
print("Prediction  :", prediction)


Loaded audio: 3.20s at 16000Hz
Ground Truth: siri what is one american dollar in japanese yen
Prediction  : SERY WHAT IS ONE AMERICAN DULL IN JAPANESIAN


## Cell 5 — `transcribe(audio_path)`

Wrapping the steps from Cell 4 into a function so we can call it in a loop.
Kept as a plain notebook function for now — this is exactly the kind of
thing that graduates to `src/asr/inference.py` later, once it's been used
across a few notebooks unchanged (per the research → refactor workflow).


In [11]:
def transcribe(audio_path, processor=processor, model=model, device=DEVICE):
    """Run baseline Wav2Vec2 CTC inference on one audio file.

    Returns the greedy-decoded transcript as a string. Returns None if the
    audio can't be loaded (missing/corrupted file), so callers can skip
    failures instead of crashing a batch run.
    """
    try:
        waveform, sr = load_audio(audio_path, target_sr=16_000)
    except Exception as e:
        print(f"Failed to load {audio_path}: {e}")
        return None

    inputs = processor(waveform, sampling_rate=sr, return_tensors="pt")
    input_values = inputs.input_values.to(device)

    with torch.no_grad():
        logits = model(input_values).logits

    predicted_ids = torch.argmax(logits, dim=-1)
    prediction = processor.batch_decode(predicted_ids)[0]
    return prediction


# quick sanity check against Cell 4's manual result
transcribe(val_set[0]["audio_path"])

'SERY WHAT IS ONE AMERICAN DULL IN JAPANESIAN'

## Cell 6 — Run on 10 Samples

In [12]:
ten_samples = [val_set[i] for i in range(10)]

for s in ten_samples:
    pred = transcribe(s["audio_path"])
    print(f"GT  : {s['transcript']}")
    print(f"Pred: {pred}")
    print("-" * 60)

GT  : siri what is one american dollar in japanese yen
Pred: SERY WHAT IS ONE AMERICAN DULL IN JAPANESIAN
------------------------------------------------------------
GT  : siri what is one american dollar in japanese yen
Pred: SERIE WAYS ONE AMERICAN BULLEN JOP IN HES YEN
------------------------------------------------------------
GT  : how many unread emails do i have
Pred: HOW MANY A RED AMALS DO I HAVE
------------------------------------------------------------
GT  : how many unread emails do i have
Pred: HOW MANY AN RED OMIN US TO I HAVE
------------------------------------------------------------
GT  : how many unread emails do i have
Pred: HOW MANY UNDRED ENOLS DO I HAVE
------------------------------------------------------------
GT  : how many unread emails do i have
Pred: HOW MANY HUNDRED GUINOTS DO I HAVE
------------------------------------------------------------
GT  : how many unread emails do i have
Pred: HOW MANY ANRITIN IMALES DO I HAVE
------------------------------

## Cell 7 — Run on 100 Samples

Random 100 (seeded), not just the first 100 — avoids any ordering bias in
the jsonl (e.g. if certain intents cluster near the top of the file).


In [13]:
eval_samples = random.sample(list(val_set), 100)

results = []
for s in tqdm(eval_samples, desc="Transcribing"):
    pred = transcribe(s["audio_path"])
    results.append({
        "audio_path": s["audio_path"],
        "ground_truth": s["transcript"],
        "prediction": pred,
        "intent": s["intent"],
        "scenario": s["scenario"],
    })

results_df = pd.DataFrame(results)
results_df.head()

Transcribing: 100%|██████████| 100/100 [00:03<00:00, 26.82it/s]


,audio_path,ground_truth,prediction,intent,scenario
0,C:\Users\ACER\OneDrive\Desktop\VoxIntel\data\r...,how long is an earth day,HOW LYNE IS ON THURSDAY,qa_factoid,qa
1,C:\Users\ACER\OneDrive\Desktop\VoxIntel\data\r...,go to youtube and play me the best wedding son...,GO TO EUCHUP AND PLAY ME THE BEST WEDDING SONG...,play_music,play
2,C:\Users\ACER\OneDrive\Desktop\VoxIntel\data\r...,what day of the week is june twenty seventh on...,DAY OF THE WEEK HE IS JUNE TWENTY SEVENTH ON T...,datetime_query,datetime
3,C:\Users\ACER\OneDrive\Desktop\VoxIntel\data\r...,what concerts are due nearby,WHAT CONSS A JUNIA BY,recommendation_events,recommendation
4,C:\Users\ACER\OneDrive\Desktop\VoxIntel\data\r...,how is coca cola stock doing today,HOWSE COT CORDE STOCK DURING,qa_stock,qa


## Cell 8 — Save Predictions

In [14]:
from pathlib import Path

output_dir = PROJECT_ROOT / "reports"
output_dir.mkdir(exist_ok=True)

output_path = output_dir / "baseline_predictions.csv"
results_df.to_csv(output_path, index=False)

print(f"Saved {len(results_df)} predictions to {output_path}")


Saved 100 predictions to c:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\baseline_predictions.csv


## Cell 9 — Calculate WER / CER

Dropping rows where transcription failed (`prediction is None`) before
scoring, and normalizing text first (lowercase, punctuation stripped) so
the metric reflects actual word errors rather than casing/punctuation
mismatches that don't matter for intent understanding.


In [15]:
from src.utils.text import normalize_text

scored_df = results_df.dropna(subset=["prediction"]).copy()
n_failed = len(results_df) - len(scored_df)
if n_failed:
    print(f"Skipping {n_failed} samples with failed transcription")

references = [normalize_text(t) for t in scored_df["ground_truth"]]
hypotheses = [normalize_text(t) for t in scored_df["prediction"]]

overall_wer = jiwer.wer(references, hypotheses)
overall_cer = jiwer.cer(references, hypotheses)

print(f"WER: {overall_wer:.4f}")
print(f"CER: {overall_cer:.4f}")

WER: 0.6357
CER: 0.3348


## Cell 10 — Analyze Failures

Per-sample WER, sorted worst-first, so we can actually read what's going
wrong rather than just staring at one aggregate number.


In [16]:
def sample_wer(ref, hyp):
    try:
        return jiwer.wer(ref, hyp)
    except Exception:
        return np.nan


scored_df["wer"] = [
    sample_wer(normalize_text(r), normalize_text(h))
    for r, h in zip(scored_df["ground_truth"], scored_df["prediction"])
]

worst = scored_df.sort_values("wer", ascending=False).head(15)

for _, row in worst.iterrows():
    print(f"WER={row['wer']:.2f}  intent={row['intent']}")
    print(f"  GT  : {row['ground_truth']}")
    print(f"  Pred: {row['prediction']}")
    print("-" * 60)

WER=1.50  intent=audio_volume_up
  GT  : audio on
  Pred: O DIUR ARM
------------------------------------------------------------
WER=1.50  intent=play_audiobook
  GT  : start play johns audiobook
  Pred: STAR A THE ENGEOS OTE BOOK
------------------------------------------------------------
WER=1.50  intent=qa_factoid
  GT  : tallest building
  Pred: TALLER YO'R DAR
------------------------------------------------------------
WER=1.40  intent=takeaway_order
  GT  : show me delivery near me
  Pred: PAD DYA I SHRE E DIVERI NEVERO
------------------------------------------------------------
WER=1.33  intent=audio_volume_up
  GT  : please turn up the speaker volume
  Pred: PLASE TRING OUT TO SPIG EN ALL IN
------------------------------------------------------------
WER=1.33  intent=news_query
  GT  : get current news on government shutdown
  Pred: GET CORRENT USED EM GO AN BE SHUT DOWN
------------------------------------------------------------
WER=1.33  intent=qa_definition
  GT  : wha

In [17]:
# Perfect vs. non-trivial errors, and whether errors cluster in certain intents
perfect = (scored_df["wer"] == 0).sum()
print(f"Perfect transcriptions: {perfect} / {len(scored_df)} ({perfect/len(scored_df)*100:.1f}%)")

print()
print("Mean WER by intent (only intents with 2+ samples in this batch):")
by_intent = scored_df.groupby("intent")["wer"].agg(["mean", "count"])
by_intent = by_intent[by_intent["count"] >= 2].sort_values("mean", ascending=False)
print(by_intent)


Perfect transcriptions: 3 / 100 (3.0%)

Mean WER by intent (only intents with 2+ samples in this batch):
                           mean  count
intent                                
play_audiobook         1.107143      2
audio_volume_up        1.033333      5
social_post            1.000000      2
takeaway_query         0.937500      2
qa_definition          0.853333      5
calendar_set           0.802424      5
recommendation_events  0.800000      2
takeaway_order         0.784848      3
qa_factoid             0.750000      6
calendar_query         0.750000      2
news_query             0.739583      4
transport_ticket       0.714286      2
social_query           0.702381      4
play_podcasts          0.666667      2
play_music             0.660985      6
calendar_remove        0.655556      3
weather_query          0.559524      6
qa_currency            0.530303      2
email_sendemail        0.529762      3
play_game              0.500000      2
music_query            0.462626      

## Notes on Failure Patterns

From manual inspection of the generated transcripts, the following trends were observed:

- Longer utterances generally contain more transcription errors than short commands.
- Named entities (person names, song titles, locations, etc.) are more likely to be misrecognized.
- Minor grammatical differences (articles, verb tense, plural forms) are occasionally introduced.
- Some words are substituted with phonetically similar alternatives.
- Short and clearly spoken commands are usually transcribed correctly.

These observations are qualitative and are based on a small sample. A more detailed error analysis will be performed after evaluating the model on the full validation set using WER and CER.

## Conclusion

- Successfully implemented a baseline Automatic Speech Recognition (ASR) pipeline using the pretrained `facebook/wav2vec2-base-960h` model.
- Verified that the model can generate transcripts directly from SLURP audio without any task-specific fine-tuning.
- The generated transcripts show that the pretrained model performs reasonably well on short, clean speech, but transcription errors are still present.
- This notebook establishes the baseline ASR system that will be used for comparison against the fine-tuned model in later experiments.
- In the next phase, the baseline will be evaluated on the complete validation split to compute reliable WER/CER metrics before fine-tuning.